# Caries verifier — tooth-crop re-ranker

The v3 disease detector localises caries well (96% of the locked-50 lesions
are hit at IoU 0.5) but ranks it poorly (35% precision). This notebook trains a
binary **tooth-crop classifier** that scores each caries detection so the
pipeline can down-weight restoration/other false positives.

Data and leakage:

| source | split | positives | hard negatives |
|---|---|---|---|
| DENTEX 4-class | 598 train / 107 val (the exact v3 split) | caries + deep caries teeth | teeth without caries |
| Romanian 14-class | 1,304 train | `CAR` teeth | `OBT`/`END`/`PRR` teeth (restorations) |

The official locked 50 is never loaded here. The verifier trains on DENTEX
train + Romanian train and is validated on the held-out 107 DENTEX images.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

WHEELS = next(Path("/kaggle/input").rglob("yolov8s.pt")).parent
os.environ["YOLO_OFFLINE"] = "True"
try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                    "--find-links", str(WHEELS), "ultralytics", "ultralytics-thop"],
                   check=True)
    import ultralytics

import torch
import torchvision
print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Verify the Kaggle account is phone-verified.")
p = torch.cuda.get_device_properties(0)
print("gpu:", p.name, f"{p.total_memory / 1e9:.1f} GB")


In [ ]:
import csv
import json
import random
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
from PIL import Image
from ultralytics import YOLO

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

QED = next(INPUT.rglob("train_quadrant_enumeration_disease.json"))
IMG_DIR = QED.parent / "xrays"
print("annotations:", QED)
print("images:", len(list(IMG_DIR.glob("*.png"))))

ROM = WORK / "romania"
if not (ROM / "train" / "images").exists():
    rom_dir = INPUT / "romania-panoramic-14class"
    zips = list(rom_dir.rglob("dataset.zip")) if rom_dir.exists() else []
    if zips:
        z = zips[0]
    else:  # fallback if the Kaggle mount is empty
        z = WORK / "romania.zip"
        if not z.exists():
            import urllib.request
            urllib.request.urlretrieve(
                "https://zenodo.org/api/records/15487430/files/"
                "panoramic_radiography_yolo_dataset_14_classes.zip/content", z)
    print("extracting", z)
    with zipfile.ZipFile(z) as fh:
        fh.extractall(ROM)
print("romanian train:", len(list((ROM / "train" / "images").glob("*.jpg"))))

TOOTH = next(INPUT.rglob("tooth_best.pt"))
print("tooth model:", TOOTH)
tooth = YOLO(str(TOOTH))

SIZE = 224
PAD = 0.10
CROPS = WORK / "crops"


In [ ]:
# ---- exact v3 split: 598 train / 107 val ----
SEED, VAL_FRAC = 42, 0.15
qed = json.load(open(QED))
imgs = {i["id"]: i for i in qed["images"]}
ids = defaultdict(set)
gt = defaultdict(list)
for ann in qed["annotations"]:
    ids[ann["image_id"]].add(ann["category_id_3"])
    img = imgs[ann["image_id"]]
    x, y, w, h = ann["bbox"]
    x1, y1 = max(0.0, x), max(0.0, y)
    x2, y2 = min(img["width"], x + w), min(img["height"], y + h)
    gt[img["file_name"]].append((ann["category_id_3"], x1, y1, x2, y2))

name_to_ids = {i["file_name"]: tuple(sorted(ids.get(i["id"], ())))
               for i in qed["images"]}


def stratified_split(names, signature_fn, val_frac=VAL_FRAC, seed=SEED):
    groups = defaultdict(list)
    for n in names:
        groups[signature_fn(n)].append(n)
    rng = random.Random(seed)
    train, val = [], []
    for key in sorted(groups):
        items = sorted(groups[key])
        rng.shuffle(items)
        n_val = max(1, int(round(len(items) * val_frac))) if len(items) > 1 else 0
        val.extend(items[:n_val])
        train.extend(items[n_val:])
    return sorted(train), sorted(val)


train_names, val_names = stratified_split(
    [i["file_name"] for i in qed["images"]], lambda n: name_to_ids[n])
print(f"train {len(train_names)} | val {len(val_names)}")
assert len(train_names) == 598 and len(val_names) == 107, "split drifted"


In [ ]:
# ---- tooth crops + labels ----
def center(b):
    return (b[0] + b[2]) / 2, (b[1] + b[3]) / 2


def inside(tb, pt):
    return tb[0] <= pt[0] <= tb[2] and tb[1] <= pt[1] <= tb[3]


def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    if inter <= 0:
        return 0.0
    aa = (a[2] - a[0]) * (a[3] - a[1])
    bb = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (aa + bb - inter)


def detec(path):
    res = tooth.predict(str(path), imgsz=1024, conf=0.25, verbose=False)[0]
    if res.boxes is None:
        return [], (0, 0)
    H, W = res.orig_shape
    boxes = [(float(x1), float(y1), float(x2), float(y2), float(c))
             for (x1, y1, x2, y2), c in zip(res.boxes.xyxy.tolist(),
                                            res.boxes.conf.tolist())]
    boxes.sort(key=lambda b: b[4], reverse=True)
    keep = []
    for b in boxes:
        if all(iou(b[:4], k[:4]) < 0.6 for k in keep):
            keep.append(b)
    return keep, (W, H)


def crop_tooth(im, tb):
    x1, y1, x2, y2 = tb
    w, h = x2 - x1, y2 - y1
    x1, x2 = x1 - PAD * w, x2 + PAD * w
    y1, y2 = y1 - PAD * h, y2 + PAD * h
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(im.width, x2), min(im.height, y2)
    if x2 - x1 < 8 or y2 - y1 < 8:
        return None
    return im.crop((x1, y1, x2, y2)).resize((SIZE, SIZE), Image.BILINEAR)


rows = []


def build_dentex(split, names):
    for k, name in enumerate(names, 1):
        path = IMG_DIR / name
        if not path.exists():
            continue
        boxes, _ = detec(path)
        if not boxes:
            continue
        im = Image.open(path).convert("RGB")
        car = [center(b[1:]) for b in gt.get(name, []) if b[0] in (1, 3)]
        for i, b in enumerate(boxes):
            tb = b[:4]
            label = int(any(inside(tb, c) for c in car))
            sub = "pos" if label else "neg"
            dst = CROPS / split / sub / f"dentex_{Path(name).stem}_{i}.jpg"
            dst.parent.mkdir(parents=True, exist_ok=True)
            crop = crop_tooth(im, tb)
            if crop is None:
                continue
            crop.save(dst, quality=92)
            rows.append({"path": str(dst), "label": label, "source": "dentex",
                         "hard": 0, "split": split, "image": name})
        if k % 100 == 0:
            print(f"  dentex {split} {k}/{len(names)}", flush=True)


build_dentex("train", train_names)
build_dentex("val", val_names)


In [ ]:
# ---- Romanian crops: CAR positives, restorations as hard negatives ----
ROM_NAMES = ["IMP", "PRR", "OBT", "END", "CAR", "BON", "IMT", "API",
             "ROT", "FUR", "APS", "ROR", "ORD", "SRD"]
rom_imgs = sorted((ROM / "train" / "images").glob("*.jpg"))
for k, path in enumerate(rom_imgs, 1):
    lab = ROM / "train" / "labels" / (path.stem + ".txt")
    if not lab.exists():
        continue
    car_norm, hard_norm = [], []
    for line in lab.read_text().splitlines():
        p = line.split()
        if len(p) != 5:
            continue
        cls, cx, cy = int(p[0]), float(p[1]), float(p[2])
        name = ROM_NAMES[cls]
        if name == "CAR":
            car_norm.append((cx, cy))
        elif name in ("OBT", "END", "PRR"):
            hard_norm.append((cx, cy))
    if not car_norm and not hard_norm:
        continue
    boxes, (W, H) = detec(path)
    if not boxes:
        continue
    car = [(cx * W, cy * H) for cx, cy in car_norm]
    hard = [(cx * W, cy * H) for cx, cy in hard_norm]
    im = Image.open(path).convert("RGB")
    for i, b in enumerate(boxes):
        tb = b[:4]
        label = int(any(inside(tb, c) for c in car))
        h = 0 if label else int(any(inside(tb, c) for c in hard))
        sub = "pos" if label else ("neg_hard" if h else "neg")
        dst = CROPS / "train" / sub / f"ro_{path.stem}_{i}.jpg"
        dst.parent.mkdir(parents=True, exist_ok=True)
        crop = crop_tooth(im, tb)
        if crop is None:
            continue
        crop.save(dst, quality=92)
        rows.append({"path": str(dst), "label": label, "source": "romania",
                     "hard": h, "split": "train", "image": path.name})
    if k % 100 == 0:
        print(f"  romania {k}/{len(rom_imgs)}", flush=True)


In [ ]:
# ---- balance: all positives/hard negatives, cap easy negatives ----
def balance(rows):
    out = []
    for split in ("train", "val"):
        r = [x for x in rows if x["split"] == split]
        pos = [x for x in r if x["label"] == 1]
        hard = [x for x in r if x["label"] == 0 and x["hard"] == 1]
        easy = [x for x in r if x["label"] == 0 and x["hard"] == 0]
        if split == "train":
            keep_easy = max(0, int(1.3 * len(pos)) - len(hard))
            random.Random(42).shuffle(easy)
            easy = easy[:keep_easy]
        out.extend(pos + hard + easy)
    return out


kept = balance(rows)
with open(CROPS / "manifest.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=["path", "label", "source", "hard", "split", "image"])
    w.writeheader()
    w.writerows(kept)
counts = defaultdict(int)
for r in kept:
    counts[(r["split"], r["label"], r["hard"])] += 1
print("kept:", dict(counts), "| total", len(kept))


In [ ]:
# ---- train ResNet18 on T4 ----
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
EPOCHS, BATCH = 6, 64


def seed_all(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


class Crops(Dataset):
    def __init__(self, rows, train):
        self.rows = rows
        self.train = train
        if train:
            self.tf = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.9, 1.1)),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(),
                transforms.Normalize(MEAN, STD),
            ])
        else:
            self.tf = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(MEAN, STD),
            ])

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        im = Image.open(r["path"]).convert("RGB")
        return self.tf(im), int(r["label"])


def auc_score(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    order = np.argsort(p)
    y = y[order]
    ranks = np.arange(1, len(y) + 1)
    n1 = y.sum()
    n0 = len(y) - n1
    if n0 == 0 or n1 == 0:
        return float("nan")
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n0 * n1))


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    ys, ps, losses = [], [], []
    for x, y in loader:
        x, y = x.cuda(), y.cuda()
        logits = model(x)
        losses.append(criterion(logits, y).item() * len(y))
        ys.extend(y.cpu().tolist())
        ps.extend(torch.softmax(logits, 1)[:, 1].cpu().tolist())
    ys, ps = np.array(ys), np.array(ps)
    pred = (ps >= 0.5).astype(int)
    tp = int(((pred == 1) & (ys == 1)).sum())
    fn = int(((pred == 0) & (ys == 1)).sum())
    tn = int(((pred == 0) & (ys == 0)).sum())
    fp = int(((pred == 1) & (ys == 0)).sum())
    return {"loss": sum(losses) / len(ys),
            "acc": float((pred == ys).mean()), "auc": auc_score(ys, ps),
            "sens": tp / max(1, tp + fn), "spec": tn / max(1, tn + fp),
            "n": int(len(ys))}


seed_all()
rows = list(csv.DictReader(open(CROPS / "manifest.csv")))
for r in rows:
    r["label"], r["hard"] = int(r["label"]), int(r["hard"])
train = [r for r in rows if r["split"] == "train"]
val = [r for r in rows if r["split"] == "val"]
print(f"train {len(train)} (pos {sum(r['label'] for r in train)}) | "
      f"val {len(val)} (pos {sum(r['label'] for r in val)})")

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).cuda()
model.fc = nn.Linear(model.fc.in_features, 2).cuda()
n0 = sum(1 for r in train if r["label"] == 0)
n1 = len(train) - n0
w = torch.tensor([len(train) / (2 * n0), len(train) / (2 * n1)], dtype=torch.float32).cuda()
criterion = nn.CrossEntropyLoss(weight=w)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
tr = DataLoader(Crops(train, True), batch_size=BATCH, shuffle=True,
                num_workers=2, drop_last=True)
va = DataLoader(Crops(val, False), batch_size=128, shuffle=False, num_workers=2)

best = -1.0
for ep in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    total, seen = 0.0, 0
    for i, (x, y) in enumerate(tr, 1):
        x, y = x.cuda(), y.cuda()
        opt.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        opt.step()
        total += loss.item() * len(y)
        seen += len(y)
        if i % 50 == 0:
            print(f"  ep {ep} {i}/{len(tr)} loss {total/seen:.4f} "
                  f"{(time.time()-t0)/i:.2f}s/it", flush=True)
    sched.step()
    m = evaluate(model, va, criterion)
    print(f"epoch {ep}: train_loss {total/max(1,seen):.4f} | val {m}", flush=True)
    if m["auc"] > best:
        best = m["auc"]
        torch.save({"state_dict": model.state_dict(), "arch": "resnet18",
                    "size": SIZE, "mean": MEAN, "std": STD,
                    "val": m, "epoch": ep}, WORK / "caries_verifier.pt")
        print(f"  saved best AUC {best:.4f}", flush=True)

print(json.dumps({"best_val_auc": best, "train_n": len(train),
                  "val_n": len(val)}, indent=2))


In [ ]:
# ---- package metrics + shrink output ----
import shutil

rows = list(csv.DictReader(open(CROPS / "manifest.csv")))
counts = defaultdict(int)
for r in rows:
    counts[f"{r['split']}_{'pos' if r['label'] == '1' else ('hard' if r['hard'] == '1' else 'neg')}"] += 1

summary = {
    "architecture": "resnet18 tooth-crop caries verifier",
    "imgsz": SIZE,
    "epochs": EPOCHS,
    "batch": BATCH,
    "train": dict(counts),
    "train_images_dentex": 598,
    "val_images_dentex": 107,
    "romanian_train_images": 1304,
    "best_val_auc": best,
    "best_val": m,
    "disease_model": "v3 (unchanged)",
    "note": "verifier only re-ranks caries detections; locked 50 not used",
}
(WORK / "caries_verifier_metrics.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

shutil.rmtree(CROPS, ignore_errors=True)
shutil.rmtree(ROM, ignore_errors=True)
(WORK / "romania.zip").unlink(missing_ok=True)
print("output:", sorted(p.name for p in WORK.iterdir()))
